# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list the `@id`s of all record sets, fields, and columns defined in the dataset schema.

In [ ]:
# List all record sets, fields, columns, and their `@id`
record_sets = list(dataset.record_sets)
print("Record Sets Found (referenced by @id):")
for rs in record_sets:
    print(f" - {rs['@id']} (name: {rs.get('name', '[no name]')})")
    fields = rs.get('fields', [])
    if fields:
        print("   Fields:")
        for f in fields:
            print(f"     - {f['@id']} (name: {f.get('name', '[no name]')})")
            columns = f.get('columns', [])
            if columns:
                print("       Columns:")
                for c in columns:
                    print(f"         - {c['@id']} (name: {c.get('name', '[no name]')})")


---
**Example: Print a few records using their record set `@id`**

Replace `<record_set_id>` with one of the record set `@id`s displayed above.

In [ ]:
# Print example records from the first record set
if len(record_sets) > 0:
    first_record_set_id = record_sets[0]['@id']
    print(f"Sample records from record set: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the `@id` fields as references throughout.

In [ ]:
dataframes = {}
record_sets_ids = [rs['@id'] for rs in record_sets]

# Load all record sets into Pandas DataFrames
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show the columns (`@id`s) and first few rows of the primary record set
if len(record_sets_ids) > 0:
    main_rs_id = record_sets_ids[0]
    print(f"Columns (fields/columns by @id) for {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference fields using their `@id`.

Choose a record set and a numeric field (by `@id`).

In [ ]:
# Example EDA: Filter by a numeric field using its `@id`
import warnings
warnings.filterwarnings('ignore')

# Choose main record set
record_set_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes.get(record_set_id, pd.DataFrame())

# Identify numeric fields (by @id)
numeric_field_id = None
for rs in record_sets:
    if rs['@id'] == record_set_id:
        for f in rs.get('fields', []):
            if f.get('dataType', '').lower() in ['integer', 'float', 'number', 'schema:integer', 'schema:float']:
                numeric_field_id = f['@id']
                break
        if numeric_field_id:
            break
# Fallback to first numeric field found in DataFrame
if not numeric_field_id and not df.empty:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

# Filter and normalize
if numeric_field_id and not df.empty:
    threshold = df[numeric_field_id].mean() if np.isfinite(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field
    group_field_id = None
    for rs in record_sets:
        if rs['@id'] == record_set_id:
            for f in rs.get('fields', []):
                if f.get('dataType', '').lower() in ['text', 'string', 'schema:text']:
                    group_field_id = f['@id']
                    break
            if group_field_id:
                break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. All references are by `@id`.

In [ ]:
# Visualize distribution of the numeric field
if numeric_field_id and not df.empty:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field_id
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides structured clinical, pathological, and molecular data for cancer survivors with second primary colorectal cancer, referenced by unique `@id`s.
- We loaded the dataset via Croissant schema and explored records, fields, and columns via their `@id` values.
- Numeric and categorical analyses can be referenced and processed using `mlcroissant`, supporting both research and clinical investigation tasks.
- The dataset enables detailed statistical and visual explorations to guide clinical stratification and policy analysis.